<a href="https://colab.research.google.com/github/NikoriakViktor/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_5/bonus/linux_devops/note_bonus_linux_devops_student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🐧 Бонус. Linux для розробника

У модулі 5 `news_hub` переїжджає в Linux: у контейнер Docker (урок 49), на сервер з Compose (50), у CI на GitHub Actions (51). Майже всі «загадкові» поломки там — звичайні речі Linux: процес не отримав сигнал, користувач не має прав на папку, сервер слухає не ту адресу, змінну середовища не передали. Сьогодні розбираємо кожну з них руками.

Colab — це Linux (Ubuntu), тож усі команди тут справжні. Команди пишемо двома способами:

- клітинка `%%bash` — виконати команди bash і подивитися вивід;
- клітинка `%%writefile скрипт.sh` **записує** скрипт, а **наступна** клітинка його запускає через помічник `sh()` і перевіряє результат (`assert`).

| Розділ | Що всередині | Приклади | Задача |
|---|---|---|---|
| 0 | робоча папка, помічник `sh()`, хто я в системі | 0.1–0.2 | |
| 1 | пайплайни: `cut`, `sort`, `uniq`, `grep`, `awk`, stdout і stderr | 1.1–1.6 | **Задача 1** |
| 2 | коди виходу, `&&` і `\|\|`, `if`, `set -e`, `pipefail` | 2.1–2.5 | |
| 3 | права: `rwx` і цифри, `chmod`, біт `x`, `umask`, інший користувач | 3.1–3.6 | |
| 4 | процеси й сигнали: `ps`, `kill`, `128 + N`, `trap`, `sh -c` і `exec`, `timeout` | 4.1–4.8 | **Задача 2** |
| 5 | порти: `127.0.0.1` чи `0.0.0.0`, `ss`, зайнятий порт | 5.1–5.5 | |
| 6 | змінні середовища: `export`, `env`, лапки і `$`, значення за замовчуванням | 6.1–6.5 | |
| 7 | скрипти, що падають вчасно: `set -u`, лапки навколо змінних, `trap EXIT` | 7.1–7.3 | **Задача 3** |
| 8 | знайди помилку: порожній бекап | 8.1–8.2 | |

**Як працювати**

- Виконуй клітинки **зверху вниз**: визначення (функція Python, текст скрипта, `%%writefile`) — в одній клітинці, **запуск — у наступній**.
- Перед **🔮 Прогнозом** спершу відповідай сам, потім розгортай відповідь.
- У Colab ти **root**. Якщо запускаєш ноутбук на своєму Linux звичайним користувачем, приклад 3.6 (інший користувач) лише покаже вивід з Colab — йому потрібні права root.
- Теорія — у книзі курсу: [Бонус. Linux](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/bonus_linux/) і [довідник з 18 розділів](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/linux/).

---
## 0. Робоча папка

### Приклад 0.1. Помічник `sh()`

Усе, що створюємо, лежить у папці `linux_lab/` (наприкінці її видаляємо). `sh(script)` виконує рядок у bash і повертає результат з `stdout`, `stderr` і `returncode` — так Python може перевірити, що надрукувала команда і чи вдалася вона.

In [ ]:
import os
import random
import subprocess
import time
from pathlib import Path

WORK = Path("linux_lab").resolve()
WORK.mkdir(exist_ok=True)
os.chdir(WORK)


def sh(script: str, check: bool = False, **kwargs) -> subprocess.CompletedProcess[str]:
    """Виконати рядок у bash у папці linux_lab; stdout і stderr — у результаті."""
    result = subprocess.run(["bash", "-c", script], capture_output=True, text=True, **kwargs)
    if check and result.returncode != 0:
        raise RuntimeError(f"код {result.returncode}: {result.stderr}")
    return result

In [ ]:
info = sh("uname -srm; whoami; pwd; bash --version | head -1")
print(info.stdout)
print("код виходу:", info.returncode)

Ядро Linux, ім'я користувача (у Colab — `root`), поточна папка і версія bash. Код виходу `0` — команди вдалися.

### Приклад 0.2. Хто я: root чи звичайний користувач

Від цього залежить, що тобі дозволено (розділ 3). `id -u` друкує номер користувача: `0` — root.

In [ ]:
IS_ROOT = os.geteuid() == 0

In [ ]:
print(sh("id").stdout.strip())
print("root:", IS_ROOT)

---
## 1. Пайплайни

Команда Linux робить одну річ і пише результат у **stdout**. `|` передає stdout однієї команди на stdin наступної. Так із маленьких програм збирають відповідь на питання, для якого окремої програми немає.

### Приклад 1.1. Журнал доступу nginx

Згенеруємо журнал у форматі nginx (такий пише сервер у `/var/log/nginx/access.log`, урок 50): IP, час, запит, код відповіді, розмір.

In [ ]:
IPS = ["10.0.0.5"] * 30 + ["172.18.0.4"] * 22 + ["192.168.1.20"] * 9 + [f"203.0.113.{i}" for i in range(1, 40)]
PATHS = ["/api/news", "/api/news/stats", "/health", "/api/admin/token", "/docs"]


def make_access_log(count: int, seed: int = 48) -> list[str]:
    random.seed(seed)
    lines = []
    for number in range(count):
        ip, path = random.choice(IPS), random.choice(PATHS)
        status = random.choice([200] * 8 + [401, 429])
        lines.append(f'{ip} - - [28/Sep/2026:10:{number // 60:02}:{number % 60:02} +0000] "GET {path} HTTP/1.1" {status} 512')
    return lines

In [ ]:
lines = make_access_log(300)
Path("access.log").write_text("\n".join(lines) + "\n")
print(sh("wc -l access.log; head -3 access.log").stdout)

300 рядків. Поля розділені пробілами: 1-ше — IP, 7-ме — шлях, 9-те — код відповіді.

### Приклад 1.2. Пайплайн крок за кроком: топ IP

Будуємо відповідь на «хто робить найбільше запитів?» по одній команді, щоб бачити, що кожна додає:

1. `cut -d' ' -f1` — лише перше поле (IP);
2. `sort` — однакові рядки поруч (без цього `uniq` не працює);
3. `uniq -c` — злити однакові сусідні рядки й порахувати;
4. `sort -rn` — за числом, від більшого;
5. `head -3` — перші три.

In [ ]:
%%bash
echo "--- cut:";            cut -d' ' -f1 access.log | head -3
echo "--- + sort | uniq -c:"; cut -d' ' -f1 access.log | sort | uniq -c | head -3
echo "--- повний пайплайн:";  cut -d' ' -f1 access.log | sort | uniq -c | sort -rn | head -3

Перевіримо пайплайн тим самим підрахунком у Python. Пайплайн записуємо в змінну (визначення), а запускаємо й порівнюємо в наступній клітинці.

In [ ]:
from collections import Counter

TOP_IPS = "cut -d' ' -f1 access.log | sort | uniq -c | sort -rn | head -3"

In [ ]:
result = sh(TOP_IPS)
print(result.stdout)
expected = Counter(line.split()[0] for line in lines).most_common(3)
got = [(parts[1], int(parts[0])) for parts in (line.split() for line in result.stdout.splitlines())]
print("Python:", expected)
assert got == expected, (got, expected)

Ті самі три IP з тими самими числами. Для гігабайтного журналу на сервері пайплайн відповідає за секунди — і нічого не треба встановлювати.

### Приклад 1.3. `grep`: лише потрібні рядки

`grep шаблон` пропускає далі лише рядки з шаблоном; `grep -c` одразу рахує; `grep -v` — навпаки, рядки **без** шаблону.

In [ ]:
%%bash
echo "запитів з кодом 429: $(grep -c '" 429 ' access.log)"
echo "запитів не до /health: $(grep -vc 'GET /health ' access.log)"
echo "хто впирається в rate limit:"
grep '" 429 ' access.log | cut -d' ' -f1 | sort | uniq -c | sort -rn | head -3

Шаблон `'" 429 '` — з лапкою й пробілами, щоб не спіймати `429` всередині часу чи розміру. Той самий пайплайн, що в 1.2, після `grep` відповідає вже на інше питання: хто отримує `429 Too Many Requests` (rate limit, урок 40).

### Приклад 1.4. `awk`: поле за номером

`awk '{print $9}'` друкує 9-те поле (код відповіді) — зручніше за `cut`, коли між полями буває кілька пробілів.

In [ ]:
%%bash
echo "коди відповідей:"
awk '{print $9}' access.log | sort | uniq -c | sort -rn
echo "найпопулярніший шлях:"
awk '{print $7}' access.log | sort | uniq -c | sort -rn | head -1

Приблизно 80 % `200`, решта — `401` і `429`, як і задано в генераторі (`[200] * 8 + [401, 429]`).

### Приклад 1.5. stdout, stderr і перенаправлення

У програми два вихідні потоки: **stdout** (результат) і **stderr** (помилки). `|` і `>` беруть лише stdout. `2>` — перенаправити stderr, `2>&1` — злити його зі stdout, `>>` — дописати в кінець файлу.

In [ ]:
%%bash
ls access.log /немає-такої-папки > out.txt 2> err.txt
echo "stdout у файлі: $(cat out.txt)"
echo "stderr у файлі: $(cat err.txt)"
echo "рядків у пайплайні без 2>&1:  $(ls access.log /немає-такої-папки 2>/dev/null | wc -l)"
echo "рядків у пайплайні з 2>&1:    $(ls access.log /немає-такої-папки 2>&1 | wc -l)"
echo "перший запуск" > journal.txt; echo "другий запуск" >> journal.txt; cat journal.txt

Помилка `ls` пішла в `err.txt`, а не в `out.txt` і не в пайплайн. Тому в CI й у `docker logs` помилки видно навіть тоді, коли stdout перенаправлено у файл. `>` перезаписує файл, `>>` — дописує.

### Приклад 1.6. `tee`: і на екран, і у файл

Скрипти деплою часто пишуть журнал і водночас показують його людині. `tee файл` копіює stdin і в файл, і далі в stdout.

In [ ]:
%%bash
grep '" 401 ' access.log | cut -d' ' -f1 | sort -u | tee unauthorized_ips.txt | wc -l
echo "у файлі: $(wc -l < unauthorized_ips.txt) рядків, перший: $(head -1 unauthorized_ips.txt)"

`sort -u` — відсортувати й лишити унікальні. Число на екрані й кількість рядків у файлі однакові: `tee` нічого не губить.

### Задача 1. Які шляхи впираються в rate limit

Допиши скрипт `top_429_paths.sh`: пайплайн, що друкує **3** шляхи (`/api/…`, `/health`, …) з найбільшою кількістю відповідей `429`, у форматі `кількість шлях` (як `uniq -c`), від найбільшого.

Підказка: `grep` (приклад 1.3) → поле з шляхом (`awk '{print $7}'`, приклад 1.4) → `sort | uniq -c | sort -rn | head -3`.

In [ ]:
%%writefile top_429_paths.sh
#!/usr/bin/env bash
# YOUR CODE HERE


In [ ]:
result = sh("bash top_429_paths.sh")
print(result.stdout, result.stderr)
assert result.stdout.strip(), "скрипт нічого не надрукував — допиши пайплайн у top_429_paths.sh"
got = [(int(count), path) for count, path in (line.split() for line in result.stdout.splitlines())]
counts = Counter(line.split()[6] for line in lines if line.split()[8] == "429")
assert len(got) == 3, f"потрібно рівно 3 рядки, а є {len(got)}"
assert all(counts[path] == count for count, path in got), f"кількість не збігається з журналом: {got}"
assert [count for count, _ in got] == sorted(counts.values(), reverse=True)[:3], "потрібні 3 найбільші, від більшого"
print("✅ Задача 1 розв'язана")

---
## 2. Коди виходу

Кожна програма завершується **кодом виходу**: `0` — успіх, інше — помилка. `$?` — код останньої команди. На кодах виходу тримається все: CI вважає крок проваленим, якщо код не 0 (урок 51), Docker показує його в `docker ps -a` (урок 49).

### Приклад 2.1. Коди простих команд і пайплайнів

🔮 **Прогноз:** який код виходу в пайплайна `false | true`? А з `set -o pipefail`?

In [ ]:
SCRIPTS = ["true", "false", "false | true", "set -o pipefail; false | true",
           "grep -q 'HTTP/1.1\" 500' access.log", "ls /немає-такої-папки", "команда-якої-немає"]

In [ ]:
for script in SCRIPTS:
    print(f"{script:<38} → код {sh(script).returncode}")

<details>
<summary>Відповідь</summary>

`false | true` — код `0`: код пайплайна — код **останньої** команди. З `pipefail` — `1`: пайплайн провалений, якщо впала будь-яка його команда. Також видно типові коди: `grep -q` без збігу — `1`, `ls` з неіснуючим шляхом — `2`, команда, якої немає, — `127`.
</details>

### Приклад 2.2. `&&` і `||`

`a && b` — `b` лише якщо `a` вдалась; `a || b` — `b` лише якщо `a` впала. Так пишуть «зроби міграції, і лише тоді запусти сервер».

In [ ]:
%%bash
mkdir -p data && echo "1. папку створено — продовжуємо"
ls /немає-такої-папки 2>/dev/null && echo "2. цього не буде видно"
ls /немає-такої-папки 2>/dev/null || echo "3. ls упав — запасний план"
grep -q '" 429 ' access.log && echo "4. у журналі є 429 — пора дивитися rate limit"

Рядок 2 не надруковано: `ls` упав, і `&&` не запустив `echo`. Рядок 3 — навпаки. Рядок 4: `grep -q` нічого не друкує, лише повертає код — його й читає `&&`.

### Приклад 2.3. `if` читає код виходу

`if команда; then …` — не про `True`/`False`, а про код: `0` — гілка `then`. Тому в `if` пишуть будь-яку команду, а `[[ … ]]` — це теж команда, що повертає 0 чи 1.

In [ ]:
%%writefile check_log.sh
#!/usr/bin/env bash
if grep -q '" 500 ' access.log; then
    echo "є помилки 500 — будимо чергового"
else
    echo "помилок 500 немає"
fi
if [[ $(grep -c '" 429 ' access.log) -gt 10 ]]; then
    echo "429 більше 10 — rate limit спрацьовує часто"
fi

In [ ]:
result = sh("bash check_log.sh")
print(result.stdout, "код:", result.returncode)

Кодів `500` генератор не створює — тож перша гілка `else`. Друга перевірка порівнює числа (`-gt` — «більше»): `[[ … ]]` повернув `0`, і `then` виконано.

### Приклад 2.4. `set -e`: зупинитись на першій помилці

У скрипті три кроки, другий падає. Без `set -e` bash виконає третій крок (сервер стартує без міграцій), з `set -e` — зупиниться.

In [ ]:
BODY = """
echo "крок 1: міграції"
ls /немає-такої-папки
echo "крок 3: запуск сервера"
"""

In [ ]:
for title, script in (("без set -e", BODY), ("з set -e", "set -e" + BODY)):
    result = sh(script)
    print(f"{title}: код {result.returncode}\n{result.stdout}")
assert "крок 3" in sh(BODY).stdout and "крок 3" not in sh("set -e" + BODY).stdout

Без `set -e` скрипт дійшов до кроку 3 і навіть завершився з кодом `0` (код останньої команди, `echo`). З `set -e` — зупинився після `ls` з кодом `2`, і CI чи Docker побачать, що щось не так.

### Приклад 2.5. Код виходу свого скрипта

`exit N` завершує скрипт з кодом N; функція bash повертає код через `return N`. Python бачить код у `returncode`.

In [ ]:
%%writefile exit_codes.sh
#!/usr/bin/env bash
check_env() {
    [[ -n "${DATABASE_URL:-}" ]] || { echo "DATABASE_URL не задано" >&2; return 3; }
    echo "DATABASE_URL=$DATABASE_URL"
}
check_env || exit $?
echo "запускаю застосунок"

In [ ]:
without = sh("bash exit_codes.sh")
with_url = sh("DATABASE_URL=sqlite:///news.db bash exit_codes.sh")
print("без змінної:", without.returncode, without.stderr.strip())
print("зі змінною: ", with_url.returncode, with_url.stdout.split())
assert without.returncode == 3 and with_url.returncode == 0

Свій код (`3`) — щоб людина чи CI одразу бачили, **яка** перевірка не пройшла. Повідомлення — у stderr (`>&2`), бо це не результат, а помилка.

---
## 3. Права й користувачі

`ls -l` показує власника, групу і права: `rwx` для власника, групи й решти. Цифрами: `r=4, w=2, x=1`, тож `640` = `rw-` власнику, `r--` групі, нічого решті. Контейнер уроку 49 працює від користувача `app` (UID 10001): код належить root і лише для читання, а в `/data` писати можна.

### Приклад 3.1. Цифри → `rwx`

Переведемо права з цифр у букви самі і звіримося зі стандартною `stat.filemode`.

In [ ]:
import stat


def mode_string(mode: int) -> str:
    """0o640 → 'rw-r-----': по три біти на власника, групу й решту."""
    letters = ""
    for shift in (6, 3, 0):
        bits = (mode >> shift) & 0o7
        letters += ("r" if bits & 4 else "-") + ("w" if bits & 2 else "-") + ("x" if bits & 1 else "-")
    return letters


MODES = {0o644: "код застосунку", 0o600: ".env з секретами", 0o755: "папка чи скрипт", 0o640: "журнал для групи"}

In [ ]:
for mode, meaning in MODES.items():
    print(f"{mode:o}  {mode_string(mode)}  {meaning}")
    assert stat.filemode(stat.S_IFREG | mode)[1:] == mode_string(mode)

`644` — усі читають, пише лише власник; `600` — лише власник; `755` — для папок і програм: заходити/запускати можуть усі.

### Приклад 3.2. `chmod` і `stat`

`chmod 600 файл` — права цифрами; `chmod u+x`, `g-w` — буквами (u — власник, g — група, o — решта). `stat -c '%a %U %n'` друкує права цифрами, власника й ім'я.

In [ ]:
%%bash
echo "JWT_SECRET=local-secret" > .env
echo "print('код застосунку')" > api.py
chmod 600 .env
chmod 644 api.py
stat -c '%a %A %U %n' .env api.py
chmod g+r .env; stat -c 'після g+r: %a %A %n' .env
chmod 600 .env

`g+r` додало групі читання (`640`), і ми повернули `600`. Саме так «чинять» права після `git clone` чи розпакування архіву.

### Приклад 3.3. Файл лише для читання — і root

🔮 **Прогноз:** файл з правами `400` (лише читання для власника). Чи вдасться власнику дописати в нього рядок? А root?

In [ ]:
Path("readonly.txt").write_text("версія 1\n")
os.chmod("readonly.txt", 0o400)
APPEND = "echo 'версія 2' >> readonly.txt && echo 'дописано' || echo 'не дописано'"

In [ ]:
print("я root:", IS_ROOT, "→", sh(APPEND).stdout.strip(), sh(APPEND).stderr.strip())
os.chmod("readonly.txt", 0o600)

<details>
<summary>Відповідь</summary>

Звичайному користувачу — ні: `Permission denied`, права діють і на власника. **Root** — так: він обходить перевірки прав на читання й запис. Саме тому застосунок у контейнері запускають не від root (`USER app`): зламаний процес root міг би переписати і код, і будь-який файл.
</details>

### Приклад 3.4. Біт `x`: запустити скрипт

`./script.sh` вимагає права на виконання (`x`) — навіть від root. `bash script.sh` — ні: тоді виконується `bash`, а скрипт він лише читає.

In [ ]:
%%writefile hello.sh
#!/usr/bin/env bash
echo "привіт з $0"

In [ ]:
for script in ("chmod 644 hello.sh; ./hello.sh", "bash hello.sh", "chmod +x hello.sh; ./hello.sh", "./немає.sh"):
    result = sh(script)
    print(f"{script:<32} → код {result.returncode}: {(result.stdout or result.stderr).strip()}")

Без `x` — код `126` («знайдено, але не можна запустити»), неіснуючий файл — `127` («не знайдено»). `#!/usr/bin/env bash` у першому рядку (shebang) каже ядру, якою програмою запускати файл. У `Dockerfile` скрипт запуску теж потребує `chmod +x` — або `COPY --chmod=755`.

### Приклад 3.5. `umask`: права нових файлів

Нові файли отримують права `666`, а папки — `777` **мінус** `umask`. Звичайний `umask 022` дає `644` і `755`; `umask 077` — лише власнику (`600`, `700`): так пишуть файли з секретами.

In [ ]:
%%bash
for mask in 022 077; do
    (umask $mask; rm -rf "new_$mask"; touch "new_$mask.txt"; mkdir "new_$mask"; stat -c "umask $mask: %a %n" "new_$mask.txt" "new_$mask")
done

Дужки `( … )` запускають команди в окремій оболонці — `umask` змінюється лише там і не впливає на решту клітинки.

### Приклад 3.6. Інший користувач: права в дії

Щоб побачити права між **різними** користувачами, потрібен root: створимо користувача `student` (UID 10001, як `app` в образі уроку 49) і виконаємо команди від його імені (`runuser -u student -- …`). Налаштуємо права так, як в образі:

- `app/api.py` — власник root, усі читають, змінює лише root (`644`);
- папка `data` — власник `student`, щоб застосунок міг туди писати;
- `.env` — лише власник читає й пише (`600`).

In [ ]:
SETUP = """
id -u student >/dev/null 2>&1 || useradd --uid 10001 --no-create-home student
mkdir -p app data
echo "print('код застосунку')" > app/api.py
chmod 755 . app
chmod 644 app/api.py && chown student data && chmod 600 .env
"""


def as_student(command: str) -> subprocess.CompletedProcess[str]:
    return sh(f"runuser -u student -- bash -c '{command}'")

In [ ]:
if IS_ROOT:
    sh(SETUP, check=True)
    print(sh("id student; stat -c '%a %U %n' app/api.py data .env").stdout)
    checks = {"читати код": "cat app/api.py", "змінити код": "echo x >> app/api.py",
              "писати в data": "touch data/news_hub.db", "читати .env": "cat .env"}
    for title, command in checks.items():
        print(f"{title:<14} → {'так' if as_student(command).returncode == 0 else 'ні'}")
    assert as_student("cat app/api.py").returncode == 0 and as_student("echo x >> app/api.py").returncode != 0
    assert as_student("touch data/news_hub.db").returncode == 0 and as_student("cat .env").returncode != 0
else:
    print("⏭ потрібні права root (у Colab вони є) — вивід з Colab нижче")

Вивід у Colab:

```text
uid=10001(student) gid=10001(student) groups=10001(student)
644 root app/api.py
755 student data
600 root .env

читати код     → так
змінити код    → ні
писати в data  → так
читати .env    → ні
```

Це й є модель контейнера `news_hub`: код читається, але не змінюється; дані пишуться лише в `/data`; секретів у файлах образу немає взагалі, а якби й були — з `600` їх не прочитав би ніхто, крім власника.

---
## 4. Процеси й сигнали

Запущена програма — **процес** з номером (PID). `ps` показує процеси, `kill -СИГНАЛ PID` надсилає сигнал:

| Сигнал | Номер | Що означає | Можна перехопити? |
|---|---|---|---|
| `SIGINT` | 2 | Ctrl+C | так |
| `SIGTERM` | 15 | «завершись, будь ласка» — `kill`, `systemctl stop`, `docker stop` | так |
| `SIGKILL` | 9 | «помри негайно» | ні |

Процес, убитий сигналом N, отримує код виходу `128 + N`: `143` після `SIGTERM`, `137` після `SIGKILL` (урок 49).

### Приклад 4.1. Процес і `ps`

In [ ]:
server = subprocess.Popen(["sleep", "300"])

In [ ]:
print("PID:", server.pid)
print(sh(f"ps -o pid,ppid,stat,cmd -p {server.pid}").stdout)

`PPID` — батьківський процес (ядро Jupyter, що запустило `sleep`), `STAT` `S` — «спить, чекає».

### Приклад 4.2. `pgrep` і `kill`

`pgrep -n -x sleep` — найновіший процес з точною назвою `sleep`; `kill -TERM PID` — надіслати йому `SIGTERM`. Python показує «вбитий сигналом 15» як код `-15`.

In [ ]:
STOP = "kill -TERM $(pgrep -n -x sleep)"

In [ ]:
sh(STOP, check=True)
code = server.wait(timeout=5)
print("код виходу:", code, "(у shell — 128 + 15 =", 128 + 15, ")")
assert code == -15

### Приклад 4.3. `128 + N` у bash

`$!` — PID останнього фонового процесу (`&`), `wait PID` — дочекатися його й отримати код.

In [ ]:
%%bash
sleep 30 & kill -TERM $!; wait $!; echo "SIGTERM → $?"
sleep 30 & kill -KILL $!; wait $!; echo "SIGKILL → $?"
sleep 30 & kill -INT  $!; wait $!; echo "SIGINT  → $?"
sleep 30 & kill -TERM $!; wait $!; echo "без сигналу код був би 0, а тут $? — тому CI бачить «упав»"

Рядок `Killed` — bash сам повідомляє в stderr, що фоновий процес убито `SIGKILL`. `143`, `137`, `130` — ці числа ти бачитимеш у `docker ps -a` (`Exited (137)`) і в журналах CI. `130` — типовий код після Ctrl+C.

### Приклад 4.4. `trap`: прибрати за собою на Ctrl+C

`trap 'команди' СИГНАЛ` — що виконати, коли прийде сигнал. Скрипт нижче створює lock-файл і на `SIGINT` (Ctrl+C) видаляє його й пише в журнал.

In [ ]:
%%writefile interruptible.sh
#!/usr/bin/env bash
trap 'echo "Ctrl+C: прибираю" >> interruptible.log; rm -f interruptible.lock; exit 130' INT
echo started > interruptible.log
touch interruptible.lock
while true; do sleep 0.1; done

In [ ]:
def run_and_signal(script: str, lock: str, signum: int) -> int:
    """Запустити bash-скрипт, дочекатися lock-файлу, надіслати сигнал, повернути код виходу."""
    process = subprocess.Popen(["bash", script])
    while not Path(lock).exists():
        time.sleep(0.05)
    process.send_signal(signum)
    return process.wait(timeout=5)

In [ ]:
import signal

code = run_and_signal("interruptible.sh", "interruptible.lock", signal.SIGINT)
print("код:", code, "| журнал:", Path("interruptible.log").read_text().splitlines(),
      "| lock лишився:", Path("interruptible.lock").exists())
assert code == 130 and not Path("interruptible.lock").exists()

Скрипт спіймав `SIGINT`, прибрав lock і сам завершився з кодом `130`, як прийнято після Ctrl+C.

### Приклад 4.5. `SIGKILL` перехопити не можна

🔮 **Прогноз:** скрипт ставить `trap` на `KILL`. Що буде з lock-файлом після `kill -9`?

In [ ]:
%%writefile unkillable.sh
#!/usr/bin/env bash
trap 'rm -f unkillable.lock' KILL
touch unkillable.lock
while true; do sleep 0.1; done

In [ ]:
code = run_and_signal("unkillable.sh", "unkillable.lock", signal.SIGKILL)
print("код:", code, "| lock лишився:", Path("unkillable.lock").exists())
assert code == -9 and Path("unkillable.lock").exists()
Path("unkillable.lock").unlink()

<details>
<summary>Відповідь</summary>

Lock лишився. `SIGKILL` ядро виконує саме, процес навіть не дізнається про нього — і `trap` на `KILL` bash просто ігнорує. Тому `docker stop` спершу дає 10 секунд на `SIGTERM`: це єдиний шанс прибрати за собою.
</details>

### Задача 2. Прибрати за собою на `SIGTERM`

Сервер перед зупинкою має закрити з'єднання, дописати журнал, видалити тимчасові файли. Допиши в `worker.sh` рядок з `trap` (як у прикладі 4.4, але для `TERM`): на `SIGTERM` скрипт має записати `cleanup done` у `worker.log`, видалити `worker.lock` і завершитись з кодом `0`.

In [ ]:
%%writefile worker.sh
#!/usr/bin/env bash
# YOUR CODE HERE
echo started > worker.log
touch worker.lock
while true; do sleep 0.1; done

In [ ]:
code = run_and_signal("worker.sh", "worker.lock", signal.SIGTERM)
print("код:", code, "| журнал:", Path("worker.log").read_text().splitlines(), "| lock:", Path("worker.lock").exists())
assert code == 0, f"код {code}: без trap bash гине від SIGTERM (-15); з trap … exit 0 — завершується сам з кодом 0"
assert "cleanup done" in Path("worker.log").read_text(), "у worker.log немає рядка cleanup done"
assert not Path("worker.lock").exists(), "worker.lock не видалено"
print("✅ Задача 2 розв'язана")

### Приклад 4.6. Через `sh -c` сигнал не доходить

Shell-форма `CMD` у Dockerfile запускає програму через `sh -c`. Перевіримо, що тоді отримує `SIGTERM`. Скрипт нижче сам по собі прибирає за собою (як у задачі 2), але запускаємо його через оболонку й надсилаємо сигнал **першому** процесу — так робить `docker stop` з PID 1.

In [ ]:
%%writefile graceful.sh
#!/usr/bin/env bash
trap 'echo "SIGTERM: прибираю" >> graceful.log; rm -f graceful.lock; exit 0' TERM
echo $$ > graceful.pid
touch graceful.lock
while true; do sleep 0.1; done

In [ ]:
def stop_first_process(command: list[str]) -> dict[str, object]:
    """Запустити command, надіслати SIGTERM першому процесу, подивитись, чи прибрав за собою graceful.sh."""
    for name in ("graceful.lock", "graceful.log", "graceful.pid"):
        Path(name).unlink(missing_ok=True)
    top = subprocess.Popen(command)
    while not Path("graceful.lock").exists():
        time.sleep(0.05)
    script_pid = int(Path("graceful.pid").read_text())
    top.send_signal(signal.SIGTERM)
    top.wait(timeout=5)
    time.sleep(0.5)
    survived = sh(f"kill -0 {script_pid}").returncode == 0
    if survived:
        os.kill(script_pid, signal.SIGKILL)
    return {"скрипт = перший процес": top.pid == script_pid, "прибрав": not Path("graceful.lock").exists(),
            "пережив SIGTERM": survived}

In [ ]:
for title, command in (("bash graceful.sh", ["bash", "graceful.sh"]),
                       ("sh -c 'bash graceful.sh; …'", ["sh", "-c", "bash graceful.sh; echo кінець"]),
                       ("sh -c 'exec bash graceful.sh'", ["sh", "-c", "exec bash graceful.sh"])):
    print(f"{title:<30} {stop_first_process(command)}")

Через `sh -c` сигнал отримала оболонка й завершилась, а скрипт його так і не побачив — «пережив» `SIGTERM` і не прибрав за собою (у контейнері через 10 с прийшов би `SIGKILL`). `exec` **замінює** оболонку програмою, тож скрипт стає першим процесом і сам отримує сигнал. Звідси правило уроку 49: exec-форма `CMD ["uvicorn", …]` або `exec` в скрипті запуску.

### Приклад 4.7. `timeout`: обмежити час команди

`timeout 1 команда` — надіслати `SIGTERM`, якщо команда працює довше 1 с; код `124` означає «перевищено час». У CI так обмежують «завислі» кроки.

In [ ]:
%%bash
timeout 1 sleep 5; echo "timeout 1 sleep 5      → код $?"
timeout 5 sleep 0.1; echo "timeout 5 sleep 0.1    → код $?"
timeout -s KILL 1 bash unkillable.sh; echo "timeout -s KILL 1 …    → код $?"
rm -f unkillable.lock

`124` — «не встиг», `0` — встиг. З `-s KILL` команда отримує `SIGKILL`, і код уже `137`.

### Приклад 4.8. Фонові процеси й `wait`

`&` запускає команду у фоні, `wait` чекає всіх фонових. Так скрипт запускає кілька незалежних кроків одночасно.

In [ ]:
%%bash
start=$(date +%s%N)
sleep 1 & sleep 1 & sleep 1 &
echo "запущено у фоні: $(jobs -p | wc -l) процеси"
wait
echo "усі завершились за $(( ($(date +%s%N) - start) / 1000000 )) мс, а не за 3000"

Три `sleep 1` паралельно — близько 1 секунди разом. Той самий принцип, що в уроці 28 з потоками, але на рівні процесів ОС.

---
## 5. Порти: `127.0.0.1` чи `0.0.0.0`

Сервер слухає **адресу й порт**. `127.0.0.1` (loopback) — з'єднання лише з тієї ж машини; `0.0.0.0` — з усіх мережевих інтерфейсів. У контейнері «та сама машина» — сам контейнер, тому uvicorn у Docker запускають з `--host 0.0.0.0`.

### Приклад 5.1. Адреса машини в мережі й вільні порти

In [ ]:
import socket
import sys


def network_ip() -> str | None:
    """Адреса цієї машини в мережі (не loopback) або None."""
    for address in sh("hostname -I 2>/dev/null").stdout.split():
        if not address.startswith("127.") and ":" not in address:
            return address
    return None


def free_port() -> int:
    with socket.socket() as sock:
        sock.bind(("127.0.0.1", 0))
        return sock.getsockname()[1]


def reachable(host: str, port: int) -> bool:
    try:
        with socket.create_connection((host, port), timeout=1):
            return True
    except OSError:
        return False

In [ ]:
EXTERNAL_IP = network_ip()
LOCAL_PORT, EVERYWHERE_PORT = free_port(), free_port()
print("адреса в мережі:", EXTERNAL_IP, "| порти:", LOCAL_PORT, EVERYWHERE_PORT)

### Приклад 5.2. Два сервери: на `127.0.0.1` і на `0.0.0.0`

Запускаємо `python3 -m http.server` двічі у фоні з різним `--bind`.

In [ ]:
def start_http_server(port: int, bind: str) -> subprocess.Popen:
    process = subprocess.Popen([sys.executable, "-m", "http.server", str(port), "--bind", bind],
                               stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    for _ in range(50):
        if reachable("127.0.0.1", port):
            break
        time.sleep(0.1)
    return process

In [ ]:
local = start_http_server(LOCAL_PORT, "127.0.0.1")
everywhere = start_http_server(EVERYWHERE_PORT, "0.0.0.0")
print("запущено:", local.pid, everywhere.pid)

### Приклад 5.3. `ss`: хто що слухає

`ss -ltn` — сокети TCP (`t`), що слухають (`l`), з номерами замість назв (`n`). Колонка `Local Address:Port` і показує, на якій адресі сервер.

In [ ]:
listening = sh(f"ss -ltn | grep -E ':({LOCAL_PORT}|{EVERYWHERE_PORT})\\b'")
print(listening.stdout or "ss недоступний: " + listening.stderr)

Один рядок з `127.0.0.1:<порт>`, інший — з `0.0.0.0:<порт>`. Так на сервері перевіряють «а чи взагалі слухає застосунок і де саме» — перше, що робити, коли «сайт не відкривається».

### Приклад 5.4. Хто до кого достукається

🔮 **Прогноз:** чотири спроби з'єднання — до кожного сервера за `127.0.0.1` і за адресою в мережі. Яка дасть `False`?

In [ ]:
if EXTERNAL_IP:
    results = {(port, host): reachable(host, port)
               for port in (LOCAL_PORT, EVERYWHERE_PORT) for host in ("127.0.0.1", EXTERNAL_IP)}
    for (port, host), ok in results.items():
        bind = "127.0.0.1" if port == LOCAL_PORT else "0.0.0.0"
        print(f"сервер на {bind:<9}, з'єднання на {host:<15} → {ok}")
    assert results == {(LOCAL_PORT, "127.0.0.1"): True, (LOCAL_PORT, EXTERNAL_IP): False,
                       (EVERYWHERE_PORT, "127.0.0.1"): True, (EVERYWHERE_PORT, EXTERNAL_IP): True}
else:
    print("⏭ у машини немає адреси в мережі, крім 127.0.0.1 — порівняти нема з чим")

<details>
<summary>Відповідь</summary>

`False` — лише сервер на `127.0.0.1` за адресою в мережі. Він приймає з'єднання тільки через loopback. У контейнері `-p 8000:8000` приводить з'єднання саме на мережевий інтерфейс контейнера — тому сервер на `127.0.0.1` усередині контейнера ззовні недоступний.
</details>

### Приклад 5.5. Порт уже зайнятий

Два процеси не можуть слухати той самий порт на тій самій адресі. Помилку `Address already in use` (errno 98) ти побачиш, коли старий сервер ще не зупинився, а новий уже стартує.

In [ ]:
def try_bind(port: int) -> str:
    with socket.socket() as sock:
        try:
            sock.bind(("0.0.0.0", port))
            return "вільний"
        except OSError as error:
            return f"зайнятий: [errno {error.errno}] {error.strerror}"

In [ ]:
print(f"порт {EVERYWHERE_PORT}:", try_bind(EVERYWHERE_PORT))
print(f"порт {free_port()}:", try_bind(free_port()))
local.terminate(); everywhere.terminate()
local.wait(); everywhere.wait()
print("після зупинки серверів:", try_bind(EVERYWHERE_PORT))

Поки сервер працює, порт зайнятий; після `terminate()` — вільний. На сервері знайти «хто зайняв порт» допомагає `ss -ltnp` (з `p` — ще й процес).

---
## 6. Змінні середовища

Процес успадковує змінні середовища від батька. Звичайна змінна shell (`NAME=value`) дочірнім процесам **не** передається; `export NAME=value` — передається; `NAME=value команда` — лише цій команді. Так `news_hub` отримує `DATABASE_URL`, `REDIS_URL`, секрети — і так їх передає Docker (`-e`, `--env-file`).

### Приклад 6.1. Три способи задати змінну

🔮 **Прогноз:** що надрукує Python у кожному рядку?

In [ ]:
READ = "python3 -c \"import os; print(os.getenv('SECRET'))\""
FORMS = [f"SECRET=abc; {READ}", f"export SECRET=abc; {READ}", f"SECRET=abc {READ}", f"SECRET=abc {READ}; {READ}"]

In [ ]:
for script in FORMS:
    print(f"{script.replace(READ, 'python3 …'):<34} → {sh(script).stdout.split()}")

<details>
<summary>Відповідь</summary>

`None`, `abc`, `abc`, `['abc', 'None']`. Змінна без `export` лишається всередині оболонки. `SECRET=abc команда` передає її лише цій одній команді — наступна її вже не бачить.
</details>

### Приклад 6.2. Середовище дочірнього процесу задає батько

З Python середовище дитини задають аргументом `env=`. `env -i` у shell — запустити з **порожнім** середовищем: так видно, що програмі справді потрібно.

In [ ]:
CHILD = [sys.executable, "-c", "import os; print(sorted(k for k in os.environ if k in ('HOME', 'PATH', 'REDIS_URL')))"]

In [ ]:
print("як у батька:      ", subprocess.run(CHILD, capture_output=True, text=True).stdout.strip())
print("env={REDIS_URL}:  ", subprocess.run(CHILD, capture_output=True, text=True,
                                          env={"REDIS_URL": "redis://redis:6379/0"}).stdout.strip())
print(sh("env -i bash -c 'echo \"PATH=[$PATH] HOME=[$HOME]\"; ls / >/dev/null && echo ls працює'").stdout)

З `env={…}` дитина бачить **лише** те, що передали, — ні `HOME`, ні `PATH`. Так працює контейнер: застосунок бачить тільки змінні з образу (`ENV`) і з `docker run -e` / `--env-file`, а не все, що є на твоєму комп'ютері. (`env -i bash` усе одно має типовий `PATH` — його bash задає сам.)

### Приклад 6.3. `$` у значенні

bcrypt-хеш пароля адміна (урок 47) містить `$`: `$2b$12$abc…`. У bash `$2` у подвійних лапках — «другий аргумент скрипта» (порожній), тож хеш зіпсується. В одинарних лапках bash нічого не підставляє. `set -a; source файл; set +a` — завантажити `.env` і експортувати все, що в ньому оголошено.

In [ ]:
HASH = "$2b$12$eBDJbnUjhmlfW0bqlm9n1u"
Path(".env.double").write_text(f'ADMIN_PASSWORD_HASH="{HASH}"\n')
Path(".env.single").write_text(f"ADMIN_PASSWORD_HASH='{HASH}'\n")
LOAD = "set -a; source {}; set +a; python3 -c \"import os; print(os.environ['ADMIN_PASSWORD_HASH'])\""

In [ ]:
double = sh(LOAD.format(".env.double")).stdout.strip()
single = sh(LOAD.format(".env.single")).stdout.strip()
print("подвійні лапки:", repr(double))
print("одинарні лапки:", repr(single))
assert single == HASH and double != HASH

З подвійними лапками від хешу лишилося `b2`: bash підставив порожні `$2`, `$1` і `$eBDJ…` (таких змінних немає), а з одинарними — точний хеш. Тому `python -m news_hub.security` друкує значення в одинарних лапках. А `docker run --env-file` лапок не знімає зовсім — для нього окремий режим `--env-file` (урок 49).

### Приклад 6.4. Значення за замовчуванням і обов'язкові змінні

`${VAR:-за_замовчуванням}` — підставити значення, якщо змінна порожня; `${VAR:?повідомлення}` — завершити скрипт з помилкою. Так скрипти деплою роблять обов'язкові змінні справді обов'язковими.

In [ ]:
%%writefile defaults.sh
#!/usr/bin/env bash
echo "REDIS_URL=${REDIS_URL:-redis://localhost:6379/0}"
echo "JWT_SECRET має ${#JWT_SECRET} символів" 2>/dev/null
: "${DATABASE_URL:?задай DATABASE_URL}"
echo "DATABASE_URL=$DATABASE_URL"

In [ ]:
for env in ("", "REDIS_URL=redis://redis:6379/0 DATABASE_URL=postgresql://db/news "):
    result = sh(f"{env}bash defaults.sh")
    print(f"[{env.strip() or 'без змінних'}] код {result.returncode}")
    print("  stdout:", result.stdout.splitlines(), "\n  stderr:", result.stderr.strip())

Без змінних `REDIS_URL` отримав значення за замовчуванням, а на `DATABASE_URL` скрипт зупинився з поясненням у stderr і ненульовим кодом. `:` — команда «нічого не роби», потрібна лише щоб bash обчислив `${…:?}`.

### Приклад 6.5. Те саме в Python: падати одразу

`news_hub` читає налаштування так само: необов'язкові — з типовим значенням, обов'язкові — з помилкою **при старті**, а не через годину на першому запиті (fail fast, урок 47).

In [ ]:
def require_env(name: str, environ: dict[str, str]) -> str:
    value = environ.get(name, "")
    if not value:
        raise RuntimeError(f"{name} не задано — застосунок не стартує")
    return value


def load_settings(environ: dict[str, str]) -> dict[str, str]:
    return {"REDIS_URL": environ.get("REDIS_URL", "redis://localhost:6379/0"),
            "DATABASE_URL": require_env("DATABASE_URL", environ)}

In [ ]:
print(load_settings({"DATABASE_URL": "sqlite+aiosqlite:////data/news_hub.db"}))
try:
    load_settings({})
except RuntimeError as error:
    print("помилка при старті:", error)

---
## 7. Скрипт, що падає вчасно

Скрипт деплою чи бекапу запускають без людини — з cron, CI, systemd. Якщо він «продовжить після помилки», наслідки побачать пізно. Тому перший рядок таких скриптів:

```bash
set -euo pipefail
```

| Опція | Що робить |
|---|---|
| `-e` | зупинитись, якщо команда завершилась з помилкою |
| `-u` | помилка, якщо змінну не задано (замість порожнього рядка) |
| `-o pipefail` | пайплайн падає, якщо впала **будь-яка** його команда, а не лише остання |

### Приклад 7.1. `set -u` і одруківка в назві змінної

Одруківка `BUIDL_DIR` замість `BUILD_DIR` без `-u` дає порожній рядок — і шлях `"$BUIDL_DIR/data"` стає `/data`. З `rm -rf` це катастрофа, тому тут лише `echo`.

In [ ]:
TYPO = 'BUILD_DIR=/tmp/news_build; echo "видаляю: $BUIDL_DIR/data"'

In [ ]:
for title, script in (("без -u", TYPO), ("з set -u", "set -u; " + TYPO)):
    result = sh(script)
    print(f"{title:<9} код {result.returncode}: {(result.stdout or result.stderr).strip()}")

Без `-u` скрипт «видалив би» `/data` у корені диска. З `-u` bash зупинився: `BUIDL_DIR: unbound variable`.

### Приклад 7.2. Лапки навколо змінних

Без лапок bash **розбиває** значення змінної на слова за пробілами. Ім'я файлу з пробілом стає двома аргументами.

In [ ]:
%%writefile quoting.sh
#!/usr/bin/env bash
count_args() { echo "$# аргумент(и): $*"; }
name="звіт за вересень.csv"
count_args $name
count_args "$name"
touch "$name"
ls $name 2>&1 | head -2
ls "$name"

In [ ]:
print(sh("bash quoting.sh").stdout)

`$name` без лапок — 3 аргументи, і `ls` шукає три різні файли. `"$name"` — один. Правило: **змінні в bash завжди в подвійних лапках**, якщо немає причини інакше.

### Приклад 7.3. `trap … EXIT`: прибрати навіть після помилки

`trap 'команди' EXIT` виконується, хоч би як скрипт завершився: нормально, через `exit` чи через `set -e`. Так прибирають тимчасові папки (`mktemp -d`), зупиняють тестові контейнери (`scripts/smoke.sh` уроку 51).

In [ ]:
%%writefile with_cleanup.sh
#!/usr/bin/env bash
set -euo pipefail
tmp=$(mktemp -d)
trap 'rm -rf "$tmp"; echo "прибрано $tmp" >&2' EXIT
echo "$tmp" > last_tmp.txt
echo "працюю в $tmp"
ls /немає-такої-папки
echo "сюди не дійде"

In [ ]:
result = sh("bash with_cleanup.sh")
tmp = Path(Path("last_tmp.txt").read_text().strip())
print("код:", result.returncode, "| stdout:", result.stdout.strip(), "\nstderr:", result.stderr.strip())
print("тимчасова папка існує:", tmp.exists())
assert result.returncode != 0 and not tmp.exists()

`set -e` зупинив скрипт на `ls`, але `trap EXIT` усе одно спрацював і видалив папку.

### Задача 3. Скрипт деплою з перевіркою аргументів

Допиши `deploy.sh`:

- перший рядок після shebang — `set -euo pipefail`;
- скрипт бере версію з першого аргументу (`$1`); якщо аргументів не рівно один (`$#`) — друкує `usage: deploy.sh VERSION` у **stderr** і завершується з кодом `2`;
- з аргументом — друкує `deploying VERSION`.

In [ ]:
%%writefile deploy.sh
#!/usr/bin/env bash
# YOUR CODE HERE


In [ ]:
sh("chmod +x deploy.sh", check=True)
no_args, with_arg, two_args = sh("./deploy.sh"), sh("./deploy.sh v49"), sh("./deploy.sh v49 extra")
print(repr(no_args.stderr), no_args.returncode, "|", repr(with_arg.stdout), with_arg.returncode)
assert no_args.returncode == 2, f"без аргументу код має бути 2, а не {no_args.returncode}"
assert "usage: deploy.sh VERSION" in no_args.stderr and no_args.stdout == "", "usage — у stderr, stdout порожній"
assert with_arg.returncode == 0 and with_arg.stdout.strip() == "deploying v49"
assert two_args.returncode == 2, "два аргументи — теж usage і код 2"
assert "set -euo pipefail" in Path("deploy.sh").read_text(), "додай set -euo pipefail"
print("✅ Задача 3 розв'язана")

---
## 8. Знайди помилку: порожній бекап

### Приклад 8.1. Скрипт, що «вдався»

Скрипт бекапу бази щоночі: `pg_dump` → `gzip` → файл. Одного разу пароль до бази змінився. Скрипт відпрацював з кодом 0, файл бекапу з'явився — і ніхто нічого не помітив. `pg_dump` тут замінено функцією, яка падає, як `pg_dump` з неправильним паролем.

In [ ]:
%%writefile backup.sh
#!/usr/bin/env bash
set -e
pg_dump() { echo "pg_dump: error: connection failed: password authentication failed" >&2; return 1; }
pg_dump news_hub | gzip > backup.sql.gz
echo "бекап готовий: $(stat -c %s backup.sql.gz) байт"

In [ ]:
result = sh("bash backup.sh")
print(result.stdout, result.stderr, "код:", result.returncode)
assert result.returncode == 0 and "бекап готовий" in result.stdout

🔮 **Прогноз:** чому `set -e` не зупинив скрипт? Що змінити **одним словом** у рядку `set -e`?

<details>
<summary>Відповідь</summary>

Код виходу пайплайна — код **останньої** команди. `gzip` успішно стиснув порожній вхід (20 байт заголовка gzip), тож `pg_dump … | gzip` «вдався», і `set -e` нічого не побачив. `set -o pipefail` робить пайплайн проваленим, якщо впала будь-яка команда: `set -eo pipefail`.
</details>

### Приклад 8.2. Виправлений скрипт

In [ ]:
FIXED = Path("backup.sh").read_text().replace("set -e\n", "set -eo pipefail\n")
Path("backup_fixed.sh").write_text(FIXED)
print(FIXED.splitlines()[1])

In [ ]:
result = sh("bash backup_fixed.sh")
print(result.stdout, result.stderr, "код:", result.returncode)
assert result.returncode != 0 and "бекап готовий" not in result.stdout

Тепер скрипт падає з кодом `1`, а cron/CI повідомить про помилку. Цей самий скрипт з'явиться в уроці 50 — для бекапу PostgreSQL з контейнера (`scripts/backup.sh`), і перший рядок там — `set -euo pipefail`.

In [ ]:
os.chdir(WORK.parent)
sh(f"rm -rf '{WORK}'")
print("linux_lab видалено:", not WORK.exists())

---
## Самоперевірка

1. Що передає `|` від однієї команди до іншої? А що не передає?
2. Який код виходу має процес, убитий `SIGKILL`, і чому його не можна перехопити?
3. Чому `./script.sh` дає код `126`, а `bash script.sh` працює?
4. Чому сервер у контейнері слухає `0.0.0.0`?
5. Чим `NAME=value` відрізняється від `export NAME=value` і від `NAME=value команда`?
6. Навіщо `pipefail`, якщо вже є `set -e`?

## Далі

- Книга курсу: [Бонус. Linux](https://nikoriakviktor.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m5/bonus_linux/) — ментальна модель, процеси й сигнали, права, мережа; довідник з 18 розділів.
- Урок 49 — Docker: контейнер — це процес Linux з власним користувачем, мережею й файловою системою.